# 📌 eValuating API and Knowledge Retrieval Agents (VAKRA)

![Topic](https://img.shields.io/badge/Topic-VAKRA-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-Agentic-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-April%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — **VAKRA** stands for eValuating API and Knowledge Retrieval Agents using multi-hop, multi-source dialogues. It is an **open-source**, executable benchmark released by IBM Research in March 2026 to measure whether AI agents can truly reason end-to-end in enterprise-like environments. **It not just answer one question, but chain multiple decisions across APIs, documents, and natural-language constraints.**</span>

---
## 1. Overview

<!-- What is this concept? 2–4 sentences that a newcomer could understand.
     Include: what problem it solves, why it exists, where it fits in the AI landscape. -->

Rather than testing isolated skills, VAKRA measures compositional reasoning across APIs and documents, using full execution traces to assess whether agents can reliably complete multi-step workflows. 
It provides an executable environment where agents interact with over 8,000 locally hosted APIs backed by real databases spanning 62 domains, along with domain-aligned document collections. Tasks require 3–7 step reasoning chains across APIs, documents, and natural-language tool-use constraints. 
It can be seen as a driving test for AI agents: not "can you steer?", but "can you navigate a full route under real conditions?"

---
## 2. How It Works

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

VAKRA has 4 progressively harder capability levels:

* **Capability 1 — Diverse API Interaction Styles** <br>
Agents must adapt to different API abstractions,some require compositional chaining, others require precise query parameterization.
* **Capability 2 — Multi-hop Reasoning over Structured APIs** <br>
Tasks require dependent reasoning chains of 1–3 APIs, where the output of one call must be correctly interpreted, transformed, and passed as input to the next. 
* **Capability 3 — Multi-hop, Multi-source with Policies** <br>
Tasks require reasoning across both unstructured documents and structured APIs, where agents must decide when to retrieve, how to ground retrieved information in downstream tool calls, and how to respect tool-use policies expressed in natural language. ibm
* **Capability 4  — Multi-turn Dialogues with Policy Adherence** <br>
Multi-turn dialogues where queries are answered by reasoning chains of 1–4 tools. For example, a three-turn dialogue where using context from the first two turns, the agent must solve a combined API + document retrieval task. 

**Evaluation is trajectory-based, not answer-based.** <br> The evaluator replays full agent trajectories against live tools and verifies every intermediate step (not just the final answer) assessing tool-use and policy adherence, exact matching of expected tool responses, and groundedness of the final answer with respect to tool outputs. 


---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **Realistic enterprise simulation** | Domains such as customer support, business intelligence, and compliance require agents to chain decisions, reconcile incompatible schemas, and respect policies expressed in natural language  |
| 🟢 | **Deterministic & verifiable** |  Locally hosted, database-backed tools ensure deterministic, verifiable responses during evaluation. |
| 🟢 | **Open-source** | Everything (data, environment, evaluator) is on GitHub and HuggingFace. |
| 🟢 | **Multi-path support**| The evaluator accepts multiple valid execution paths, which is essential for real workflows where there is more than one correct route to a solution.  |
| 🔴 | **Heavy infrastructure requirements** | You need 8 GB+ of memory allocated to Docker/Podman and about 35 GB of disk space for the benchmark data. Not lightweight. |
| 🔴 | **Self-hosted only** | There is no hosted cloud version. You run everything locally via Docker containers.|
| 🔴 | **Narrow scope** | VAKRA focuses on tool-calling and retrieval agents. It does not evaluate creativity, general knowledge, or code generation. |
| 🔴 | **Very new** | Released in March 2026, the community and ecosystem are still forming. The leaderboard has few entries so far. |


---
## 4. How to use

> **Goal:** Quick explanation to test VAKRA using hockey example provided on the IBM repository

In [ ]:
# ---------------------------------------------------------------------------
# Step 1 — Install & Download
# ---------------------------------------------------------------------------

git clone https://github.com/IBM/vakra.git
cd vakra

python3 -m venv .venv
source .venv/bin/activate

pip install -e ".[init]"
pip install -r requirements_benchmark.txt

make download        # downloads ~35 GB of benchmark data

# ---------------------------------------------------------------------------
# Step 2 — Build & Start the Environment
# ---------------------------------------------------------------------------

make build
docker compose up -d

# ---------------------------------------------------------------------------
# Step 3 — Explore Available Tools (Optional but Recommended)
# ---------------------------------------------------------------------------

cd tools_explorer
uvicorn app:app --reload --port 7861

Open http://localhost:7861 in your browser to browse all 8,000+ tools, inspect their schemas, and test individual calls before running any agent.

# ---------------------------------------------------------------------------
# Step 4 — Run a Benchmark (Example: Claude on Hockey data)
# ---------------------------------------------------------------------------

python benchmark_runner.py \
  --capability_id 2 \
  --domain hockey \
  --provider anthropic \
  --model claude-sonnet-4-6

  
Or with no API key using Ollama:

ollama pull llama3.1:8b

python benchmark_runner.py \
  --capability_id 1 \
  --domain card_games \
  --max-samples-per-domain 5 \
  --provider ollama \
  --model llama3.1:8b

# ---------------------------------------------------------------------------
# Step 5 — Evaluate Output
# ---------------------------------------------------------------------------

python validate_output.py --capability 2 # using Anthropic
python validate_output.py --capability 1 # using ollama 

Concrete Reasoning Example

Imagine a task: "Which hockey teams played in the 2018 playoffs, and what was the total count?"

The agent must:

1. Call get_hockey_teams(season=2018) → gets a list
2. Pass the result to compute_data_count(key_name="team_id") → gets 16
3. Return a grounded answer: "16 teams"

VAKRA verifies both steps, not just the final answer. If the agent skipped step 1 and guessed, it would fail the trajectory check even if the number happened to be right.


---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **VAKRA is one of the first to test end-to-end compositional reasoning in realistic enterprise workflows**
- **Trajectory verification is the key innovation.** 
- **The 4-capability structure mirrors real enterprise complexity.** 
- **It supports your LLM of choice.** 
- **It is a signal for production deployment decisions.** 

</div>